In [ ]:
"""
Loads the RAW (pre-windowing) signals from RespiBan and Empatica and exports
them as plain, human-readable CSV files — each with a sample-index column
placed at the END (rather than pandas' default index at the start), so the
actual signal columns are the first thing you see.

Requires windowing.py to be in the same folder (reuses its loaders).
"""

import os
import pandas as pd
 

import sys, os
sys.path.append(os.path.abspath("../preprocessing/common_filter/raw_window.py"))
from raw_window import load_respiban_txt, load_empatica_csv, sliding_window
# ----------------------------------------------------------------------------
# RESPIBAN
# ----------------------------------------------------------------------------
def export_respiban_csv(respiban_txt_path, out_path):
    """
    Loads the RespiBan .txt file, keeps only the meaningful sensor columns
    (drops nSeq/DI housekeeping columns), and writes a CSV with columns:
        ECG, EDA, EMG, TEMP, ACC_x, ACC_y, ACC_z, RESPIRATION, index
    """
    df, fs = load_respiban_txt(respiban_txt_path)

    sensor_cols = [c for c in
                   ["ECG", "EDA", "EMG", "TEMP", "ACC_x", "ACC_y", "ACC_z", "RESPIRATION"]
                   if c in df.columns]
    df = df[sensor_cols].copy()

    df["index"] = range(len(df))  # sample index, placed as the last column

    df.to_csv(out_path, index=False)
    print(f"[respiban] fs={fs} Hz, {len(df)} samples -> saved to {out_path}")
    return df


# ----------------------------------------------------------------------------
# EMPATICA
# ----------------------------------------------------------------------------
def export_empatica_csv(empatica_dir, out_dir):
    """
    Loads every Empatica signal (ACC, BVP, EDA, HR, TEMP, IBI) and writes
    ONE readable CSV per signal (they can't be merged into a single table
    since each has a different sampling rate / length). Each CSV gets an
    'index' column at the end.

    ACC.csv        -> acc_x, acc_y, acc_z, index
    BVP/EDA/HR/TEMP -> value, index
    IBI.csv        -> time_since_start_sec, ibi_sec, index
    """
    os.makedirs(out_dir, exist_ok=True)
    signal_files = {
        "acc": "ACC.csv",
        "bvp": "BVP.csv",
        "eda": "EDA.csv",
        "hr": "HR.csv",
        "temp": "TEMP.csv",
        "ibi": "IBI.csv",
    }

    for name, fname in signal_files.items():
        fpath = os.path.join(empatica_dir, fname)
        if not os.path.exists(fpath):
            print(f"[skip] {fpath} not found")
            continue

        out_path = os.path.join(out_dir, f"empatica_{name}.csv")

        if name == "ibi":
            t, vals, start_ts, _ = load_empatica_csv(fpath)
            df = pd.DataFrame({"time_since_start_sec": t, "ibi_sec": vals})
            df["index"] = range(len(df))
            df.to_csv(out_path, index=False)
            print(f"[empatica] ibi: {len(df)} events -> saved to {out_path}")
            continue

        data, fs, start_ts = load_empatica_csv(fpath)

        if name == "acc":
            df = pd.DataFrame(data, columns=["acc_x", "acc_y", "acc_z"])
        else:
            df = pd.DataFrame({name: data})

        df["index"] = range(len(df))
        df.to_csv(out_path, index=False)
        print(f"[empatica] {name}: fs={fs} Hz, {len(df)} samples -> saved to {out_path}")


# ----------------------------------------------------------------------------
# EXAMPLE USAGE
# ----------------------------------------------------------------------------
if __name__ == "__main__":
    EMPATICA_DIR = "../data/raw/WESAD/S2/S2_E4_Data"       # folder containing ACC.csv, EDA.csv, ...
    RESPIBAN_TXT = "../data/raw/WESAD/S2/respiban.txt"

    out_dir = "raw_readable_csv"
    os.makedirs(out_dir, exist_ok=True)

    if os.path.isfile(RESPIBAN_TXT):
        export_respiban_csv(RESPIBAN_TXT, os.path.join(out_dir, "respiban.csv"))

    if os.path.isdir(EMPATICA_DIR):
        export_empatica_csv(EMPATICA_DIR, out_dir)

    print("\nDone. Readable CSVs saved to:", out_dir)


ModuleNotFoundError: No module named 'windowing'